## Import Libraries for Heatmap generation

1) Plot without clustering
2) Split each feature into 3 different layers:
	1) Spatial, Texture, Intensity
	2) Cell Region
	3) SER-edge or STAR feature

#### z-scores: blue=low, white=mean, red=high.

In [9]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import numpy as np
import re
from matplotlib.patches import Patch

In [11]:
rng = np.random.default_rng(seed=42)

# Normalise for Batch 1

In [12]:
data_path = 'data/Raw_Data_Batch1_LAB1_CTX2.csv'
counts = pd.read_csv(data_path)
counts = counts.drop(columns=[
                   	"Row",
                    "Column",
                    "Timepoint",
                    "Field",
                    "Object No",
                    "X",
                    "Y",
                    "Bounding Box",
                    "Batch",
                    "Nuclei - Object No in Nuclei_all"
                   ])
# Double check here
counts = counts.drop(columns=[
    'Position X [Âµm]',
	'Position Y [Âµm]',
    'Compound',
    'Concentration',
    'Cell Count'
])
print("counts (before): ", counts.shape)
counts = counts.fillna(0)
print("counts (after): ", counts.shape)

counts (before):  (20682, 1322)
counts (after):  (20682, 1322)


In [13]:
counts.head()

,Cell Type,Cellline ID,Lysosome Intensity - Nucleus region Mean,Lysosome Intensity - Nucleus region StdDev,Lysosome Intensity - Nucleus region Median,Lysosome Intensity - Nucleus region Maximum,Lysosome Intensity - Nucleus region Minimum,Lysosome Intensity - Nucleus region Sum,Lysosome Intensity - Nucleus region CV [%],Lysosome Intensity - Nucleus region Quantile 90%,...,Mitochondria Texture - Outer region SER Valley 1 px,Mitochondria Texture - Outer region SER Saddle 1 px,Mitochondria Texture - Outer region SER Bright 1 px,Mitochondria Texture - Outer region SER Dark 1 px,Mitochondria Texture - Outer region Haralick Correlation 2 px,Mitochondria Texture - Outer region Haralick Contrast 2 px,Mitochondria Texture - Outer region Haralick Sum Variance 2 px,Mitochondria Texture - Outer region Haralick Homogeneity 2 px,Mitochondria Texture - Outer region Gabor Min 2 px w2,Mitochondria Texture - Outer region Gabor Max 2 px w2
0,Control,RM3.5,1964.49,473.458,2001,2964,747,1092250,24.1009,2558,...,0.062429,0.055786,0.060275,0.080337,0.692103,0.179664,0.246843,0.195137,0.000550,0.004230
1,Control,RM3.5,2582.91,555.290,2683,3507,1303,883355,21.4986,3261,...,0.056252,0.055994,0.062334,0.067784,0.661050,0.187395,0.229586,0.210063,0.000543,0.004167
2,Control,RM3.5,3105.96,806.695,2904,6924,1006,2963080,25.9725,4097,...,0.047268,0.060245,0.063949,0.054644,0.708571,0.196272,0.287673,0.203688,0.000483,0.003577
3,Control,RM3.5,1677.06,552.588,1557,3253,779,753001,32.9498,2438,...,0.084388,0.079462,0.080827,0.100119,0.681948,0.367140,0.485384,0.179421,0.000649,0.005440
4,Control,RM3.5,3321.61,796.328,3175,9111,1523,4354630,23.9741,4097,...,0.057507,0.053676,0.054210,0.068928,0.770866,0.165035,0.318869,0.227847,0.000484,0.003702


In [14]:
counts[['Lysosome Spatial - Outer region Profile 3/5', 'Lysosome Spatial - Outer region Profile 3/5 SER-Dark', 'Lysosome Spatial - Outer region Profile 3/5 SER-Ridge', 'Lysosome Spatial - Outer region Profile 3/5 SER-Spot', 'Lysosome Spatial - Outer region Profile 3/5 SER-Edge', 'Mitochondria Spatial - Outer region Profile 3/5', 'Mitochondria Spatial - Outer region Profile 3/5 SER-Dark', 'Mitochondria Spatial - Outer region Profile 3/5 SER-Ridge', 'Mitochondria Spatial - Outer region Profile 3/5 SER-Spot', 'Mitochondria Spatial - Outer region Profile 3/5 SER-Edge']]

,Lysosome Spatial - Outer region Profile 3/5,Lysosome Spatial - Outer region Profile 3/5 SER-Dark,Lysosome Spatial - Outer region Profile 3/5 SER-Ridge,Lysosome Spatial - Outer region Profile 3/5 SER-Spot,Lysosome Spatial - Outer region Profile 3/5 SER-Edge,Mitochondria Spatial - Outer region Profile 3/5,Mitochondria Spatial - Outer region Profile 3/5 SER-Dark,Mitochondria Spatial - Outer region Profile 3/5 SER-Ridge,Mitochondria Spatial - Outer region Profile 3/5 SER-Spot,Mitochondria Spatial - Outer region Profile 3/5 SER-Edge
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
20677,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20678,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20679,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20680,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Calculate Aggregate level values

1. Get the columns that are area, roundness and ratio and non-SER features
2. Drop those columns
3. Sort the remaining columns to get their labels

In [15]:
counts_without_cellline = counts.drop(columns=['Cellline ID'])
cell_type_agg_df = counts_without_cellline.groupby(['Cell Type']).mean()
cell_type_agg_df

,Lysosome Intensity - Nucleus region Mean,Lysosome Intensity - Nucleus region StdDev,Lysosome Intensity - Nucleus region Median,Lysosome Intensity - Nucleus region Maximum,Lysosome Intensity - Nucleus region Minimum,Lysosome Intensity - Nucleus region Sum,Lysosome Intensity - Nucleus region CV [%],Lysosome Intensity - Nucleus region Quantile 90%,Lysosome Intensity - Nucleus region Contrast,General Dimension - Nucleus region Area µm²,...,Mitochondria Texture - Outer region SER Valley 1 px,Mitochondria Texture - Outer region SER Saddle 1 px,Mitochondria Texture - Outer region SER Bright 1 px,Mitochondria Texture - Outer region SER Dark 1 px,Mitochondria Texture - Outer region Haralick Correlation 2 px,Mitochondria Texture - Outer region Haralick Contrast 2 px,Mitochondria Texture - Outer region Haralick Sum Variance 2 px,Mitochondria Texture - Outer region Haralick Homogeneity 2 px,Mitochondria Texture - Outer region Gabor Min 2 px w2,Mitochondria Texture - Outer region Gabor Max 2 px w2
Cell Type,,,,,,,,,,,,,,,,,,,,,
Control,2415.820238,1068.276350,2115.358462,9310.675777,1123.228563,2.418000e+06,47.227396,3635.190569,0.133404,97.131325,...,0.066482,0.051583,0.044348,0.074617,0.730109,0.257773,0.476910,0.270425,0.000529,0.004023
LRRK2,2021.875701,1112.009004,1704.602551,9416.777551,785.303571,1.903551e+06,52.428042,3175.903571,0.221412,82.328004,...,0.060666,0.043065,0.040697,0.065448,0.723399,0.241826,0.531648,0.285185,0.000421,0.003122
PRKN,2076.053908,1952.357934,1410.677438,17065.391393,625.548616,2.735305e+06,94.056915,4003.532532,-0.133413,120.537051,...,0.083197,0.051577,0.039248,0.092716,0.748071,0.339158,0.710430,0.309782,0.000661,0.004936
SNCA,1951.625292,1429.647893,1496.251453,12805.726463,768.235761,2.151740e+06,73.047671,3316.071678,0.014999,102.527258,...,0.068528,0.047660,0.036714,0.074893,0.738253,0.372691,0.711612,0.341564,0.000580,0.004187


In [16]:
cell_line_agg_df = counts.groupby(['Cellline ID', 'Cell Type']).mean()
print("cell_line_agg (before): ", cell_line_agg_df.shape)
cell_line_agg_df = cell_line_agg_df.dropna()
print("cell_line_agg (after): ", cell_line_agg_df.shape)

cell_line_agg (before):  (13, 1320)
cell_line_agg (after):  (13, 1320)


In [17]:
cell_line_agg_df

,,Lysosome Intensity - Nucleus region Mean,Lysosome Intensity - Nucleus region StdDev,Lysosome Intensity - Nucleus region Median,Lysosome Intensity - Nucleus region Maximum,Lysosome Intensity - Nucleus region Minimum,Lysosome Intensity - Nucleus region Sum,Lysosome Intensity - Nucleus region CV [%],Lysosome Intensity - Nucleus region Quantile 90%,Lysosome Intensity - Nucleus region Contrast,General Dimension - Nucleus region Area µm²,...,Mitochondria Texture - Outer region SER Valley 1 px,Mitochondria Texture - Outer region SER Saddle 1 px,Mitochondria Texture - Outer region SER Bright 1 px,Mitochondria Texture - Outer region SER Dark 1 px,Mitochondria Texture - Outer region Haralick Correlation 2 px,Mitochondria Texture - Outer region Haralick Contrast 2 px,Mitochondria Texture - Outer region Haralick Sum Variance 2 px,Mitochondria Texture - Outer region Haralick Homogeneity 2 px,Mitochondria Texture - Outer region Gabor Min 2 px w2,Mitochondria Texture - Outer region Gabor Max 2 px w2
Cellline ID,Cell Type,,,,,,,,,,,,,,,,,,,,,
01-060 C9,PRKN,2397.114860,1293.805829,2005.592018,9983.676275,926.840355,1.875812e+06,52.855346,3842.866962,0.232157,67.946315,...,0.065262,0.047551,0.053530,0.070469,0.743751,0.266924,0.642336,0.249505,0.000416,0.003145
09-090 C18,PRKN,1946.967211,2127.857417,1206.144343,18769.161198,506.857630,2.820884e+06,105.234601,3998.820154,-0.220533,130.065552,...,0.087697,0.052232,0.036871,0.097903,0.748054,0.361059,0.746866,0.322500,0.000715,0.005327
111450-107,Control,2515.278619,1090.267033,2192.716625,8956.351385,1179.583753,2.261869e+06,43.335420,3816.617128,0.163312,78.931746,...,0.066566,0.054189,0.048099,0.075258,0.733000,0.241992,0.452307,0.244027,0.000524,0.003997
11302-101,Control,2809.234488,958.642255,2568.322977,8513.619942,1404.583815,2.631833e+06,36.016652,3949.494942,0.219692,85.389591,...,0.068447,0.051690,0.045649,0.076666,0.723425,0.253551,0.452969,0.254463,0.000530,0.004026
11555-104,SNCA,2480.385462,1103.076554,2175.409756,9469.023415,1121.036098,2.414524e+06,43.835067,3753.430244,0.215947,86.073801,...,0.067242,0.054169,0.047672,0.075259,0.737148,0.268140,0.525461,0.255511,0.000503,0.003764
11556-110,SNCA,1640.041258,1645.694736,1094.231889,15174.415912,594.461190,2.056489e+06,91.588414,3051.448900,-0.131112,115.168152,...,0.070784,0.045249,0.029053,0.076389,0.732905,0.444822,0.825245,0.401549,0.000662,0.004694
11557-103,SNCA,2354.915873,1110.718500,2019.607656,9069.968421,936.360766,2.175820e+06,46.841877,3670.063158,0.250218,81.263283,...,0.063117,0.048410,0.048634,0.070107,0.755164,0.261815,0.557979,0.248481,0.000412,0.003101
11576-101,LRRK2,1470.152495,1268.766833,1056.077670,10608.302358,471.406380,1.837719e+06,69.614077,2754.846047,0.026012,98.174964,...,0.051716,0.033858,0.027448,0.055842,0.720464,0.206805,0.515336,0.349847,0.000440,0.003197
14555-107,LRRK2,2378.571557,1120.186412,2069.684814,9633.982808,986.262178,2.028495e+06,45.832643,3530.806590,0.298809,75.417705,...,0.065156,0.049727,0.049404,0.070891,0.738615,0.279009,0.558219,0.245275,0.000419,0.003110


In [18]:
control_agg_batch_1 = cell_type_agg_df.loc['Control']
normalised_agg_batch_1 = cell_line_agg_df/control_agg_batch_1
normalised_agg_batch_1

,,Lysosome Intensity - Nucleus region Mean,Lysosome Intensity - Nucleus region StdDev,Lysosome Intensity - Nucleus region Median,Lysosome Intensity - Nucleus region Maximum,Lysosome Intensity - Nucleus region Minimum,Lysosome Intensity - Nucleus region Sum,Lysosome Intensity - Nucleus region CV [%],Lysosome Intensity - Nucleus region Quantile 90%,Lysosome Intensity - Nucleus region Contrast,General Dimension - Nucleus region Area µm²,...,Mitochondria Texture - Outer region SER Valley 1 px,Mitochondria Texture - Outer region SER Saddle 1 px,Mitochondria Texture - Outer region SER Bright 1 px,Mitochondria Texture - Outer region SER Dark 1 px,Mitochondria Texture - Outer region Haralick Correlation 2 px,Mitochondria Texture - Outer region Haralick Contrast 2 px,Mitochondria Texture - Outer region Haralick Sum Variance 2 px,Mitochondria Texture - Outer region Haralick Homogeneity 2 px,Mitochondria Texture - Outer region Gabor Min 2 px w2,Mitochondria Texture - Outer region Gabor Max 2 px w2
Cellline ID,Cell Type,,,,,,,,,,,,,,,,,,,,,
01-060 C9,PRKN,0.992257,1.211115,0.948110,1.072283,0.825157,0.775770,1.119167,1.057129,1.740250,0.699530,...,0.981647,0.921818,1.207045,0.944410,1.018684,1.035499,1.346869,0.922638,0.785347,0.781786
09-090 C18,PRKN,0.805924,1.991860,0.570184,2.015875,0.451251,1.166619,2.228253,1.100030,-1.653118,1.339069,...,1.319117,1.012572,0.831399,1.312066,1.024578,1.400684,1.566051,1.192564,1.351099,1.324170
111450-107,Control,1.041170,1.020585,1.036570,0.961944,1.050173,0.935430,0.917591,1.049908,1.224192,0.812629,...,1.001272,1.050510,1.084566,1.008583,1.003959,0.938776,0.948410,0.902382,0.989152,0.993611
11302-101,Control,1.162849,0.897373,1.214131,0.914393,1.250488,1.088434,0.762622,1.086462,1.646812,0.879115,...,1.029565,1.002066,1.029339,1.027459,0.990845,0.983618,0.949800,0.940974,1.000109,1.000759
11555-104,SNCA,1.026726,1.032576,1.028388,1.017007,0.998048,0.998562,0.928170,1.032526,1.618743,0.886159,...,1.011427,1.050117,1.074948,1.008592,1.009641,1.040218,1.101803,0.944850,0.949845,0.935743
11556-110,SNCA,0.678876,1.540514,0.517280,1.629787,0.529243,0.850491,1.939307,0.839419,-0.982816,1.185695,...,1.064706,0.877200,0.655114,1.023738,1.003829,1.725630,1.730399,1.484879,1.251156,1.166851
11557-103,SNCA,0.974789,1.039730,0.954735,0.974147,0.833633,0.899843,0.991837,1.009593,1.875641,0.836633,...,0.949380,0.938475,1.096642,0.939549,1.034316,1.015680,1.169987,0.918853,0.778958,0.770864
11576-101,LRRK2,0.608552,1.187677,0.499243,1.139370,0.419689,0.760016,1.474019,0.757827,0.194989,1.010745,...,0.777899,0.656370,0.618913,0.748380,0.986789,0.802274,1.080572,1.293691,0.831828,0.794592
14555-107,LRRK2,0.984581,1.048592,0.978409,1.034724,0.878060,0.838914,0.970467,0.971285,2.239874,0.776451,...,0.980055,0.964014,1.113998,0.950058,1.011650,1.082381,1.170492,0.906996,0.791762,0.772975


## Export as csv

In [19]:
normalised_agg_batch_1.to_csv("batch_1_normalised_cellline_agg.csv", index=True)

# Make the whole thing into a function

In [29]:
def export_normalise_cellline_agg(counts_path, batch_no):
    # Get the data in a dataframe
    data_path = counts_path
    counts = pd.read_csv(f"data/{data_path}")
    counts = counts.drop(columns=[
                        "Row",
                        "Column",
                        "Timepoint",
                        "Field",
                        "Object No",
                        "X",
                        "Y",
                        "Bounding Box",
                        "Batch",
                        "Nuclei - Object No in Nuclei_all"
                    ])
    # Double check here
    counts = counts.drop(columns=[
        'Position X [Âµm]',
        'Position Y [Âµm]',
        'Compound',
        'Concentration',
        'Cell Count'
    ])
    print("counts (before): ", counts.shape)
    counts = counts.fillna(0)
    print("counts (after): ", counts.shape)

    # Calculate aggregate level data for cell type
    counts_without_cellline = counts.drop(columns=['Cellline ID'])
    cell_type_agg_df = counts_without_cellline.groupby(['Cell Type']).mean()

    # Calculate the aggregate level data for cellline
    cell_line_agg_df = counts.groupby(['Cellline ID', 'Cell Type']).mean()
    print("cell_line_agg (before): ", cell_line_agg_df.shape)
    cell_line_agg_df = cell_line_agg_df.dropna()
    print("cell_line_agg (after): ", cell_line_agg_df.shape)   

    # Normalise it using the average of controls for that particular batch
    control_agg = cell_type_agg_df.loc['Control']
    normalised_agg = cell_line_agg_df/control_agg 

    # Export as csv
    normalised_agg.to_csv(f"data/cell_line_agg_data/batch_{str(batch_no)}_normalised_cellline_agg.csv", index=True)


In [30]:
from pathlib import Path
directory_path = Path('data')

# Recursively find all files
for file in directory_path.rglob('*.csv'):
    if file.is_file():
        file_name_parts = re.split(r'_', file.name)
        batch_no = file_name_parts[2][-1]
        lab_no = file_name_parts[3]
        # print(batch_no)
        # print(file_name_parts)

        if lab_no == 'LAB1':
            export_normalise_cellline_agg(str(file.name), batch_no)

counts (before):  (23803, 1322)
counts (after):  (23803, 1322)
cell_line_agg (before):  (13, 1320)
cell_line_agg (after):  (13, 1320)
counts (before):  (21575, 1322)
counts (after):  (21575, 1322)
cell_line_agg (before):  (12, 1320)
cell_line_agg (after):  (12, 1320)
counts (before):  (20682, 1322)
counts (after):  (20682, 1322)
cell_line_agg (before):  (13, 1320)
cell_line_agg (after):  (13, 1320)
